# **Preprocessing for Bridge Grades: App Communications Calculations Data**

This notebook prepares the input data used to generate 4 foundational resources in the Bridge Grades methodology:

- **Source C:** *Sums the outcome_bipartisan column for each representative from the communication file.*  


These resources serve as the basis for computing individual Bridge Grades by analyzing patterns of bipartisan legislative collaboration in the U.S. Congress.

The datasets processed here correspond to the **119th U.S. Congress** and were downloaded from the public data portal at [Americas Political Pulse](https://americaspoliticalpulse.com/data/).

- **Data includes:** All forms of public communication by legislators in the House and Senate are automatically collected, including:

  - Floor speeches

  - Newsletters

  - Press releases

  - Tweets/X
- **Download instructions:**
Go to the button "Congressional Rhetoric" and download by clicking on the button that says "Download" in the card "All Classified Statements".

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Install and import required libraries
import pandas as pd
import numpy as np
import glob

In [3]:
# Load raw data comunications
df= pd.read_csv('/content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2026_06_02/Raw_C_2026_06_02/rhetoric-all.csv',low_memory=False)

# Read in the 119th Congress data with bioguide ids
files = sorted(glob.glob('/content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2026_06_02/Raw_Others_2026_06_02/119th_Congress_20260602.csv'))
latest = files[-1]
df_119 = pd.read_csv(latest)

## **Bipartisanship processing**

This helper function transforms the raw 119th Congress bioguide spreadsheet into a clean lookup by promoting the header row, extracting and reformatting full names, and retaining only `bioguide_id` and `full_name`.  

In [4]:
df_119.columns

Index(['bioguide_id', 'Name', 'first_name', 'middle_name', 'last_name',
       'nickname', 'Chamber', 'State', 'District', 'Party', 'start_year',
       'end_year', 'update_date', 'imageUrl'],
      dtype='object')

In [5]:
df_bioguide = df_119.copy()
df_bioguide['full_name'] = df_bioguide['first_name'] + ' ' + df_bioguide['last_name']
df_bioguide = df_bioguide[['bioguide_id', 'full_name']]
df_bioguide.head()

,bioguide_id,full_name
0,F000483,Laura Friedman
1,V000139,Matt Van Epps
2,B001288,Cory Booker
3,S001150,Adam Schiff
4,A000383,Alan Armstrong


## **Processing and Aggregation of APP Communication Data**

This function distills the raw APP communications data into per‐legislator summaries by counting total messages, aggregating personal attacks, bipartisan outcomes, and policy mentions, and then computing corresponding percentage rates for 2025.  

In [6]:
df.columns

Index(['id', 'source', 'source_id', 'bioguide_id', 'text', 'policy',
       'policy_area', 'policy_explanation', 'attack_personal', 'attack_policy',
       'attack_target', 'attack_explanation', 'outcome_bipartisanship',
       'outcome_creditclaiming', 'date', 'bipartisanship_explanation',
       'creditclaiming_explanation', 'source_url', 'classified', 'attack_type',
       'extreme_label', 'last_updated', 'extreme_target', 'first_name',
       'last_name', 'state', 'type', 'tweet_id', 'url'],
      dtype='object')

In [7]:
df.date.min()

'2023-01-01'

In [8]:
df = df[df['date'] >= '2025-01-01'].copy()

In [9]:
len(df[df['bioguide_id']=='A000055'])

572

In [10]:
def clean_communications_app(df):
    # Keep only necessary columns
    df = df[['bioguide_id', 'attack_personal', 'outcome_bipartisanship', 'policy', 'first_name', 'last_name']].copy()

    # Count total number of communications per legislator
    df['communication_count'] = df.groupby('bioguide_id')['bioguide_id'].transform('count')

    # Create full_name column from first and last name
    df['full_name'] = df['first_name'].str.strip() + ' ' + df['last_name'].str.strip()

    # Aggregate total flags by legislator
    df = df.groupby(['bioguide_id', 'full_name', 'communication_count'], as_index=False).agg({
        'outcome_bipartisanship': 'sum'
    })

    return df


# Clean and sort APP communication dataset for year 2025
df_comm_app_2025 = (
    clean_communications_app(df)
    .sort_values(by='bioguide_id', ascending=True)
    .reset_index(drop=True)
)

In [11]:
df_comm_app_2025.head(20)

,bioguide_id,full_name,communication_count,outcome_bipartisanship
0,A000055,Robert Aderholt,572,35
1,A000148,Jake Auchincloss,1073,112
2,A000369,Mark Amodei,391,30
3,A000370,Alma Adams,396,20
4,A000371,Pete Aguilar,334,19
5,A000372,Rick Allen,920,52
6,A000375,Jodey Arrington,1235,50
7,A000379,Mark Alford,2640,127
8,A000380,Gabe Amo,1724,95
9,A000381,Yassamin Ansari,364,11


In [12]:
# Export
df_comm_app_2025.to_csv(
    '/content/drive/Shareddrives/Data Archive/Bridge_Grades/Output_Files/Temp_Files/Temp_C/Temp_C_v1.0.1_2026_06_02.csv',
    index=False
)

In [13]:
# Check that all legislators are present
df_comm_app_2025.shape[0]

533